# Week 11: Improved RAG — Measuring Query Expansion

Same logic as `compare_retrieval.py`. Requires `LLM_API_KEY` and `LLM_MODEL` in a `.env` file (see `.env.example`) — query expansion calls a real LLM to generate rephrasings. Retrieval itself needs no API key. Reuses the persistent collection Week 9 built (`data/processed/chroma`) — run `examples/week-09/build_passage_index.py` first if you haven't already.

Verified for real against this course's own sample passages before being written: basic retrieval scores **4/6 (66.7%)** on `data/sample/eval_questions.json`; with query expansion, **6/6 (100%)**.

In [ ]:
import json
from functools import partial
from pathlib import Path

from ai_finance_course.query_expansion import retrieve_with_expansion
from ai_finance_course.retrieval_eval import hit_rate
from ai_finance_course.vector_store import get_or_create_collection, query_collection

QUESTIONS_PATH = Path("data/sample/eval_questions.json")
PERSIST_PATH = Path("data/processed/chroma")
COLLECTION_NAME = "sample_passages"

questions = json.loads(QUESTIONS_PATH.read_text(encoding="utf-8"))
collection = get_or_create_collection(PERSIST_PATH, COLLECTION_NAME)
len(questions)

## Diagnose: Which Questions Does Basic Retrieval Miss?

In [ ]:
for question in questions:
    result = query_collection(collection, question["query"], n_results=1)[0]
    ok = result["metadata"]["ticker"] == question["expected_ticker"]
    status = "OK  " if ok else "MISS"
    print(f"{status} {question['query']!r} -> got {result['metadata']['ticker']}, expected {question['expected_ticker']}")

## Basic Retrieval Hit Rate

In [ ]:
basic_rate = hit_rate(collection, questions, query_collection, n_results=1)
print(f"Basic retrieval hit rate: {basic_rate:.1%}")

## Set Up the Real LLM Call

In [ ]:
import os

import httpx
from dotenv import load_dotenv

ANTHROPIC_MESSAGES_URL = "https://api.anthropic.com/v1/messages"


def call_llm(prompt: str) -> str:
    """The one Anthropic-specific piece; retrieve_with_expansion() itself is provider-agnostic."""
    with httpx.Client(timeout=60.0) as client:
        response = client.post(
            ANTHROPIC_MESSAGES_URL,
            headers={
                "x-api-key": os.environ["LLM_API_KEY"],
                "anthropic-version": "2023-06-01",
                "content-type": "application/json",
            },
            json={
                "model": os.environ["LLM_MODEL"],
                "max_tokens": 1024,
                "messages": [{"role": "user", "content": prompt}],
            },
        )
        response.raise_for_status()
        data = response.json()
        for block in data["content"]:
            if block["type"] == "text":
                return block["text"]
        raise ValueError(f"No text block in response: {data}")


load_dotenv()

## Improved (Query-Expanded) Retrieval Hit Rate

In [ ]:
improved_retrieve = partial(retrieve_with_expansion, generate=call_llm)
improved_rate = hit_rate(collection, questions, improved_retrieve, n_results=1)
print(f"Improved (expanded) hit rate: {improved_rate:.1%}")
print(f"Improvement: {improved_rate - basic_rate:+.1%}")